# 05 — Exploratory review of unmatched OpenSanctions entities

This notebook supports a **limited, transparent manual review** of unmatched Netherlands-assigned OpenSanctions entities selected in notebook 04.

It contains:

1. all records associated with Dutch national institutions selected for review (for example, Netherlands Senate and House of Representatives memberships); and
2. the reproducible sample of 15 EveryPolitician records created in notebook 04.

This notebook is exploratory only. It does **not** modify the benchmark, matching algorithm, or reported coverage measures. Manual review classifications are saved separately as an audit trail.

The reviewed cases and resulting classifications are specific to the current live-source benchmark snapshot created in notebooks 01 and 02.


## 1. Setup and file paths

Run notebook 04 before this notebook. The required review-selection file is created by notebook 04.


In [1]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

# The notebook is expected to be stored in:
# <project folder>/notebooks/
PROJECT_DIR = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_DIR / "data" / "output"

REVIEW_SELECTION_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_entity_scope_review_sample.csv"
)

MANUAL_REVIEW_TEMPLATE_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_manual_review_template.csv"
)

MANUAL_REVIEW_SUMMARY_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_manual_review_summary.csv"
)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 30)

print("Project directory:", PROJECT_DIR)
print("Review-selection input:", REVIEW_SELECTION_PATH)
print("Manual-review template:", MANUAL_REVIEW_TEMPLATE_PATH)


Project directory: C:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean
Review-selection input: C:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_entity_scope_review_sample.csv
Manual-review template: C:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_manual_review_template.csv


## 2. Read the selected review cases

The helper function supports the encodings used by the preceding notebooks. It stops with a clear error if notebook 04 has not yet created the review-selection file.


In [2]:
def read_csv_with_fallback_encodings(file_path):
    """Read a CSV using common encodings used in this project."""
    if not file_path.exists():
        raise FileNotFoundError(
            "Required file was not found:\n"
            f"{file_path}\n\n"
            "Run notebook 04 from top to bottom before running this notebook."
        )

    possible_encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1252",
        "latin1"
    ]

    last_error = None

    for encoding in possible_encodings:
        try:
            return pd.read_csv(
                file_path,
                encoding=encoding
            )
        except UnicodeDecodeError as error:
            last_error = error

    raise last_error


def write_csv_safely(dataframe, output_path):
    """Save a UTF-8 CSV, using a timestamped fallback if it is locked."""
    try:
        dataframe.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    except PermissionError:
        timestamp = datetime.now(
            timezone.utc
        ).strftime("%Y%m%d_%H%M%S")

        fallback_path = output_path.with_name(
            f"{output_path.stem}_{timestamp}{output_path.suffix}"
        )

        dataframe.to_csv(
            fallback_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "Warning: the intended file was open or locked. "
            f"Saved fallback file: {fallback_path.name}"
        )

        return fallback_path


review_selection_df = read_csv_with_fallback_encodings(
    REVIEW_SELECTION_PATH
)

print("Selected review rows:", len(review_selection_df))
print(
    "Unique OpenSanctions entities:",
    review_selection_df["opensanctions_id"].nunique()
)

print("\nSelection groups:")
display(
    review_selection_df[
        "review_selection"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("review_selection")
    .reset_index(name="records")
)


Selected review rows: 21
Unique OpenSanctions entities: 21

Selection groups:


,review_selection,records
0,reproducible_random_sample,15
1,all_high_medium_fuzzy_candidates,4
2,all_dutch_national_institution_records,2


## 3. Build or reload the manual-review template

The template is stored separately from the notebook 04 outputs. On the first run, it is created with blank review fields. On later runs, existing manual decisions are preserved by OpenSanctions identifier.

Use the following values consistently where possible:

- `out_of_scope`: the record is not part of the empirically defined Dutch national benchmark;
- `historical_or_timing_difference`: the record concerns a former role or a difference in source timing;
- `name_or_alias_variation`: the person appears relevant but is missed because of name representation;
- `potential_benchmark_omission`: the record appears in scope and may require benchmark review;
- `insufficient_information`: the available evidence is not enough to decide;
- `other`: another explanation applies.


In [3]:
manual_review_columns = [
    "manual_classification",
    "manual_scope_assessment",
    "manual_likely_explanation",
    "manual_relevant_to_benchmark",
    "manual_evidence_source",
    "manual_review_notes",
    "reviewer",
    "review_date"
]

# Keep all selection metadata from notebook 04.
manual_review_df = review_selection_df.copy()

# Create a practical link field for browser-based manual checks.
manual_review_df["opensanctions_entity_url"] = (
    "https://www.opensanctions.org/entities/"
    + manual_review_df["opensanctions_id"].astype(str)
    + "/"
)

# Set defaults for a first review run.
for column in manual_review_columns:
    if column not in manual_review_df.columns:
        manual_review_df[column] = pd.NA

manual_review_df["manual_classification"] = (
    manual_review_df["manual_classification"]
    .fillna("needs_manual_review")
)

manual_review_df["manual_scope_assessment"] = (
    manual_review_df["manual_scope_assessment"]
    .fillna("needs_manual_review")
)

manual_review_df["manual_relevant_to_benchmark"] = (
    manual_review_df["manual_relevant_to_benchmark"]
    .fillna("unclear")
)

# Preserve manual changes when the template has already been edited.
if MANUAL_REVIEW_TEMPLATE_PATH.exists():
    existing_review_df = read_csv_with_fallback_encodings(
        MANUAL_REVIEW_TEMPLATE_PATH
    )

    preserved_columns = [
        column
        for column in manual_review_columns
        if column in existing_review_df.columns
    ]

    if "opensanctions_id" in existing_review_df.columns:
        existing_manual_df = existing_review_df[
            ["opensanctions_id"] + preserved_columns
        ].drop_duplicates(
            subset=["opensanctions_id"]
        )

        manual_review_df = (
            manual_review_df
            .drop(columns=manual_review_columns)
            .merge(
                existing_manual_df,
                on="opensanctions_id",
                how="left"
            )
        )

        for column in manual_review_columns:
            if column not in manual_review_df.columns:
                manual_review_df[column] = pd.NA

        manual_review_df["manual_classification"] = (
            manual_review_df["manual_classification"]
            .fillna("needs_manual_review")
        )

        manual_review_df["manual_scope_assessment"] = (
            manual_review_df["manual_scope_assessment"]
            .fillna("needs_manual_review")
        )

        manual_review_df["manual_relevant_to_benchmark"] = (
            manual_review_df["manual_relevant_to_benchmark"]
            .fillna("unclear")
        )

        print("Existing manual-review decisions were preserved.")
    else:
        print(
            "Warning: the existing template has no opensanctions_id "
            "column. A fresh template will be created."
        )

manual_review_df = (
    manual_review_df
    .drop_duplicates(subset=["opensanctions_id"])
    .sort_values(
        [
            "review_selection",
            "entity_scope_classification",
            "opensanctions_primary_name"
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)

template_path_written = write_csv_safely(
    manual_review_df,
    MANUAL_REVIEW_TEMPLATE_PATH
)

print("Manual-review template saved to:")
print(template_path_written)


Existing manual-review decisions were preserved.
Manual-review template saved to:
C:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_manual_review_template_20260622_155832.csv


## 4. Review all Dutch national-institution cases

These records are the most relevant to inspect because they have a Netherlands Senate or Netherlands House of Representatives source membership but did not receive a deterministic benchmark link.

Use the `opensanctions_entity_url`, source memberships, fuzzy candidate, and review fields to document your decision. Do not modify the benchmark directly from this notebook.


In [4]:
display_columns = [
    "opensanctions_primary_name",
    "opensanctions_id",
    "opensanctions_entity_url",
    "non_generic_dataset_memberships",
    "best_fuzzy_score",
    "best_fuzzy_benchmark_person_name",
    "best_fuzzy_benchmark_categories",
    "fuzzy_review_priority",
    "manual_classification",
    "manual_scope_assessment",
    "manual_relevant_to_benchmark",
    "manual_likely_explanation",
    "manual_review_notes"
]

dutch_institution_cases_df = manual_review_df[
    manual_review_df["review_selection"]
    .eq("all_dutch_national_institution_records")
].copy()

print(
    "Dutch national-institution cases to review:",
    len(dutch_institution_cases_df)
)

display(
    dutch_institution_cases_df[
        [
            column
            for column in display_columns
            if column in dutch_institution_cases_df.columns
        ]
    ]
)


Dutch national-institution cases to review: 2


,opensanctions_primary_name,opensanctions_id,opensanctions_entity_url,non_generic_dataset_memberships,best_fuzzy_score,best_fuzzy_benchmark_person_name,best_fuzzy_benchmark_categories,fuzzy_review_priority,manual_classification,manual_scope_assessment,manual_relevant_to_benchmark,manual_likely_explanation,manual_review_notes
0,Meryem Kılıç-Karaaslan,Q117235809,https://www.opensanctions.org/entities/Q117235809/,Netherlands Senate,95.0,Meryem Karaaslan,senate_members,high_priority_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
1,Ton van Kesteren,Q29054517,https://www.opensanctions.org/entities/Q29054517/,Netherlands Senate,85.5,A.E.H. van der Voort Maarschalk,high_judiciary_hr,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN


## 5. Review the reproducible EveryPolitician sample

This sample was selected with `random_state=42` in notebook 04. It supports a transparent, manageable check of whether the source primarily represents historical records, name variants, or potential benchmark omissions.


In [5]:
everypolitician_sample_df = manual_review_df[
    manual_review_df["review_selection"]
    .eq("reproducible_random_sample")
].copy()

print(
    "EveryPolitician records in reproducible sample:",
    len(everypolitician_sample_df)
)

display(
    everypolitician_sample_df[
        [
            column
            for column in display_columns
            if column in everypolitician_sample_df.columns
        ]
    ]
)


EveryPolitician records in reproducible sample: 15


,opensanctions_primary_name,opensanctions_id,opensanctions_entity_url,non_generic_dataset_memberships,best_fuzzy_score,best_fuzzy_benchmark_person_name,best_fuzzy_benchmark_categories,fuzzy_review_priority,manual_classification,manual_scope_assessment,manual_relevant_to_benchmark,manual_likely_explanation,manual_review_notes
6,Agnes Mulder,Q94239,https://www.opensanctions.org/entities/Q94239/,MySociety's EveryPolitician Legislators,85.50,A.E.B. ter Heide,high_judiciary_hr,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
7,Albert de Vries,Q1194985,https://www.opensanctions.org/entities/Q1194985/,MySociety's EveryPolitician Legislators,85.50,C. de Kruif **,high_judiciary_cbb,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
8,Danai van Weerdenburg,Q19801955,https://www.opensanctions.org/entities/Q19801955/,MySociety's EveryPolitician Legislators,85.50,A.J. van Doesum,high_judiciary_hr,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
9,Eric Smaling,Q12013388,https://www.opensanctions.org/entities/Q12013388/,MySociety's EveryPolitician Legislators,61.54,Eric Kemperman,senate_members,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
10,Evelyn Wever-Croes,Q23038705,https://www.opensanctions.org/entities/Q23038705/,MySociety's EveryPolitician Legislators,85.50,A.E.H. van der Voort Maarschalk,high_judiciary_hr,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
11,Frank Futselaar,Q28861221,https://www.opensanctions.org/entities/Q28861221/,MySociety's EveryPolitician Legislators,85.50,C.F.E. van Olden-Smit,high_judiciary_crvb,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
12,Gerrit Schotte,Q3091857,https://www.opensanctions.org/entities/Q3091857/,MySociety's EveryPolitician Legislators,85.50,E.F. Faase,high_judiciary_hr,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
13,Henk van Gerven,Q746203,https://www.opensanctions.org/entities/Q746203/,MySociety's EveryPolitician Legislators,85.50,A.E.H. van der Voort Maarschalk,high_judiciary_hr,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
14,Lenny Geluk-Poortvliet,Q28861156,https://www.opensanctions.org/entities/Q28861156/,MySociety's EveryPolitician Legislators,85.50,C.C.W. Lange,high_judiciary_cbb; high_judiciary_crvb,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN
15,Marit Maij,Q617451,https://www.opensanctions.org/entities/Q617451/,European Parliament Members; MySociety's EveryPolitician Legislators,72.00,Mariëtte Patijn,house_members,standard_review,needs_manual_review,needs_manual_review,unclear,NaN,NaN


## 6. How to complete the review

1. Open `opensanctions_nl_manual_review_template.csv` in Excel.
2. Fill the manual fields for the selected cases.
3. Save the file with the same filename and close Excel before rerunning this notebook.
4. Rerun the next cell to generate an auditable summary.

Suggested evidence sources include the linked OpenSanctions entity page, the cited source dataset, and the official Dutch institutional source used for the benchmark.


In [8]:
# Re-save the current template after any in-notebook edits.
template_path_written = write_csv_safely(
    manual_review_df,
    MANUAL_REVIEW_TEMPLATE_PATH
)

print("Current template saved to:")
print(template_path_written)

print(
    "\nImportant: edits made in Excel are loaded when you rerun "
    "Section 3. This notebook will preserve them by OpenSanctions ID."
)


Current template saved to:
C:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_manual_review_template.csv

Important: edits made in Excel are loaded when you rerun Section 3. This notebook will preserve them by OpenSanctions ID.


## 7. Summarise manual-review decisions

Run this section after editing and reloading the template. The output is suitable for a short methodological appendix or for informing the interpretation of reverse validation. It should not be used as a statistical benchmark-completeness estimate.


In [7]:
manual_review_summary_df = (
    manual_review_df
    .groupby(
        [
            "review_selection",
            "manual_classification",
            "manual_scope_assessment",
            "manual_relevant_to_benchmark"
        ],
        dropna=False
    )
    .agg(
        reviewed_entities=(
            "opensanctions_id",
            "nunique"
        ),
        example_names=(
            "opensanctions_primary_name",
            lambda values: "; ".join(
                sorted(
                    set(values)
                )[:5]
            )
        )
    )
    .reset_index()
    .sort_values(
        [
            "review_selection",
            "reviewed_entities"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

summary_path_written = write_csv_safely(
    manual_review_summary_df,
    MANUAL_REVIEW_SUMMARY_PATH
)

print("Manual-review summary:")
display(manual_review_summary_df)

print("\nSaved summary to:")
print(summary_path_written)


Manual-review summary:


,review_selection,manual_classification,manual_scope_assessment,manual_relevant_to_benchmark,reviewed_entities,example_names
0,all_dutch_national_institution_records,needs_manual_review,needs_manual_review,unclear,2,Meryem Kılıç-Karaaslan; Ton van Kesteren
1,all_high_medium_fuzzy_candidates,needs_manual_review,needs_manual_review,unclear,4,Ans van den Berg; Henk Bakker; Jan de Ruiter; Patrick van den Brink
2,reproducible_random_sample,needs_manual_review,needs_manual_review,unclear,15,Agnes Mulder; Albert de Vries; Danai van Weerdenburg; Eric Smaling; Evelyn Wever-Croes



Saved summary to:
C:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_manual_review_summary.csv
